In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS dbr_dev.weather_gold;

In [0]:
%sql
CREATE OR REPLACE TABLE dbr_dev.weather_gold.gold_last_24h
LOCATION 'abfss://dataweather@dlspl21databricks.dfs.core.windows.net/gold/tables/gold_last_24h'
AS
SELECT
    city,
    event_time,
    temperature,
    wind_speed,
    wind_direction,
    air_quality,
    pm10,
    pm2_5
FROM dbr_dev.weather_silver.unified_weather
-- We only filter events from the last 24 hours
WHERE event_time >= current_timestamp() - INTERVAL 24 HOURS;

In [0]:
%sql
CREATE OR REPLACE TABLE dbr_dev.weather_gold.gold_last_7_days
LOCATION 'abfss://dataweather@dlspl21databricks.dfs.core.windows.net/gold/tables/gold_last_7_days'
AS
SELECT
    city,
    DATE(event_time) AS event_date,
    ROUND(AVG(temperature), 1) AS avg_temp,
    MAX(temperature) AS max_temp,
    MIN(temperature) AS min_temp,
    ROUND(AVG(wind_speed), 1) AS avg_wind,
    ROUND(AVG(air_quality), 0) AS avg_air_quality
FROM dbr_dev.weather_silver.unified_weather
-- We only filter events from the last 7 days
WHERE event_time >= current_timestamp() - INTERVAL 7 DAYS
GROUP BY city, DATE(event_time);

In [0]:
%sql
CREATE OR REPLACE TABLE dbr_dev.weather_gold.gold_city_profiles
LOCATION 'abfss://dataweather@dlspl21databricks.dfs.core.windows.net/gold/tables/gold_city_profiles'
AS
SELECT
    city,
    COUNT(*) AS total_measurements_recorded,
    MIN(event_time) AS data_collected_since,
    ROUND(AVG(temperature), 1) AS historical_avg_temp,
    MAX(temperature) AS all_time_high_temp,
    MIN(temperature) AS all_time_low_temp
FROM dbr_dev.weather_silver.unified_weather
GROUP BY city;